# Quisquid

Speaker-attributed transcription of audio or video recordings.

Transcription d'enregistrements audio ou vidéo avec attribution des locuteurs.

### English

To use it, click Connect at the top right, then click the small arrow at the top left of the code cell to run it. Scroll down to the output of the cell, where the language choice, the configuration and the file upload take place.

For better performance, use a T4 GPU (small arrow next to Connect, Change runtime type, select T4 GPU).

### Français

Pour l'utiliser, cliquez sur Connecter en haut à droite, puis sur la petite flèche en haut à gauche du script pour lancer la cellule. Scrollez en bas de la page pour voir le résultat d'exécution de la cellule, c'est là où tout le reste (importation, paramétrage) se fera.

Pour de meilleures performances, connectez-vous en GPU T4. (Petite flèche à côté de Connecter, Modifier le type d'exécution, sélectionner GPU T4)

Copyright (c) 2026 Alexandre Mahieu. PolyForm Noncommercial License 1.0.0, described at the end of the notebook. Licence PolyForm Noncommercial 1.0.0, dont les conditions figurent en bas du notebook.

In [ ]:
import importlib.util
import os
import subprocess
import sys

TEXTS = {
    "installing": {"fr": "Installation des dépendances (2 à 4 minutes lors de la première exécution)...",
                   "en": "Installing dependencies (2 to 4 minutes on the first run)..."},
    "install_done": {"fr": "Installation terminée.",
                     "en": "Installation complete."},
    "old_transformers": {"fr": "Une ancienne version de Transformers est déjà chargée en mémoire. Redémarrez la session "
                               "(Exécution > Redémarrer la session) puis relancez la cellule.",
                         "en": "An older version of Transformers is already loaded in memory. Restart the session "
                               "(Runtime > Restart session), then run the cell again."},
    "deps_present": {"fr": "Dépendances déjà installées",
                     "en": "Dependencies already installed"},
    "installing_cublas": {"fr": "Installation de cuBLAS pour CUDA 12 (requis par faster-whisper)...",
                          "en": "Installing cuBLAS for CUDA 12 (required by faster-whisper)..."},
    "cublas_missing": {"fr": "ATTENTION : cuBLAS 12 introuvable, la transcription sur GPU échouera.",
                       "en": "WARNING: cuBLAS 12 not found, transcription on the GPU will fail."},
    "cublas_loaded": {"fr": "cuBLAS 12 chargé depuis {path}",
                      "en": "cuBLAS 12 loaded from {path}"},
    "imports_ok": {"fr": "Tous imports réussis",
                   "en": "All imports successful"},
    "no_gpu": {"fr": "ATTENTION : aucun GPU détecté. Activez-le dans Exécution > Modifier le type d'exécution > GPU T4.",
               "en": "WARNING: no GPU detected. Enable it in Runtime > Change runtime type > T4 GPU."},
    "no_gpu_slow": {"fr": "Sans GPU, la transcription sera très lente.",
                    "en": "Without a GPU, transcription will be very slow."},

    "download_text": {"fr": "Télécharger le texte",
                      "en": "Download the text"},
    "download_spreadsheet": {"fr": "Télécharger le tableur",
                             "en": "Download the spreadsheet"},
    "download_file": {"fr": "Télécharger le fichier",
                      "en": "Download the file"},
    "download_all": {"fr": "Télécharger tout",
                     "en": "Download all"},
    "size_kb": {"fr": "{value} Ko",
                "en": "{value} KB"},
    "size_mb": {"fr": "{value} Mo",
                "en": "{value} MB"},
    "decimal_separator": {"fr": ",",
                          "en": "."},
    "thousands_separator": {"fr": " ",
                            "en": ","},
    "speaker_one": {"fr": "locuteur",
                    "en": "speaker"},
    "speaker_many": {"fr": "locuteurs",
                     "en": "speakers"},
    "word_one": {"fr": "mot",
                 "en": "word"},
    "word_many": {"fr": "mots",
                  "en": "words"},
    "file_one": {"fr": "fichier",
                 "en": "file"},
    "file_many": {"fr": "fichiers",
                  "en": "files"},
    "transcribed_one": {"fr": "transcrit",
                        "en": "transcribed"},
    "transcribed_many": {"fr": "transcrits",
                         "en": "transcribed"},
    "drive_ok": {"fr": "Copie enregistrée dans MyDrive/transcriptions.",
                 "en": "Copy saved in MyDrive/transcriptions."},
    "drive_failed": {"fr": "La copie sur Drive a échoué.",
                     "en": "The copy to Drive failed."},
    "drive_none": {"fr": "Aucune copie sur Drive.",
                   "en": "No copy on Drive."},
    "drive_download_advice": {"fr": "Téléchargez les fichiers avant de fermer cet onglet. Les liens restent valides tant que "
                                    "l'onglet est ouvert, même si la session Colab s'arrête.",
                              "en": "Download the files before closing this tab. The links remain valid while the tab "
                                    "is open, even if the Colab session stops."},
    "no_speech_transcribed": {"fr": "aucune parole transcrite",
                              "en": "no speech transcribed"},
    "hint_no_speech": {"fr": "Vérifiez que le fichier contient de la parole audible et que le son n'est pas coupé.",
                       "en": "Check that the file contains audible speech and that the sound is not muted."},
    "hint_ffmpeg": {"fr": "Le fichier n'a pas pu être lu. Convertissez-le en MP3 ou en WAV, puis relancez-le.",
                    "en": "The file could not be read. Convert it to MP3 or WAV, then run it again."},
    "hint_cuda": {"fr": "Une bibliothèque CUDA requise par Whisper est introuvable. Redémarrez la session "
                        "(Exécution > Redémarrer la session), relancez la cellule, puis transmettez le message "
                        "si l'erreur se reproduit.",
                  "en": "A CUDA library required by Whisper cannot be found. Restart the session "
                        "(Runtime > Restart session), run the cell again, then report the message "
                        "if the error occurs again."},
    "hint_oom": {"fr": "La mémoire du GPU est saturée. Redémarrez la session ou choisissez le modèle turbo.",
                 "en": "The GPU memory is full. Restart the session or choose the turbo model."},
    "hint_default": {"fr": "Relancez ce fichier seul. Si l'erreur se reproduit, transmettez le message affiché dans la console.",
                     "en": "Run this file again on its own. If the error occurs again, report the message shown in "
                           "the console."},
    "result_title": {"fr": "Transcription de {name}",
                     "en": "Transcription of {name}"},
    "position": {"fr": "({index} sur {total})",
                 "en": "({index} of {total})"},
    "speakers_undistinguished": {"fr": "locuteurs non distingués",
                                 "en": "speakers not distinguished"},
    "result_meta": {"fr": "{minutes} min, langue {language}, {speakers}, {words}",
                    "en": "{minutes} min, language {language}, {speakers}, {words}"},
    "diarization_failed_note": {"fr": "La diarisation a échoué. Le texte est complet mais n'indique pas qui parle. "
                                      "Relancez ce fichier pour obtenir l'attribution des locuteurs.",
                                "en": "Diarisation failed. The text is complete but does not indicate who is speaking. "
                                      "Run this file again to obtain speaker attribution."},
    "failure_title": {"fr": "Échec de la transcription de {name}",
                      "en": "Transcription of {name} failed"},
    "technical_detail": {"fr": "Détail technique : {detail}",
                         "en": "Technical detail: {detail}"},
    "batch_title": {"fr": "Lot terminé : {done} sur {total} {files} {transcribed}",
                    "en": "Batch complete: {done} of {total} {files} {transcribed}"},
    "batch_archive": {"fr": "transcriptions_lot.zip",
                      "en": "transcriptions_batch.zip"},
    "item_transcribed": {"fr": "{name} : transcrit",
                         "en": "{name}: transcribed"},
    "item_failed": {"fr": "{name} : échec",
                    "en": "{name}: failed"},
    "drive_enabled": {"fr": "Sauvegarde Drive activée : {path}",
                      "en": "Drive backup enabled: {path}"},
    "drive_mount_failed": {"fr": "Impossible de monter Drive : {error}",
                           "en": "Could not mount Drive: {error}"},
    "drive_copied": {"fr": "→ Drive : {name}",
                     "en": "→ Drive: {name}"},
    "drive_copy_failed": {"fr": "! Echec copie Drive pour {path} : {error}",
                          "en": "! Drive copy failed for {path}: {error}"},

    "progress_transcription": {"fr": "Transcription",
                               "en": "Transcription"},
    "loading_diarizer": {"fr": "Chargement du modèle de diarisation...",
                         "en": "Loading the diarisation model..."},
    "diarization_gpu_retry": {"fr": "Echec de la diarisation sur GPU ({error}), nouvel essai sur CPU...",
                              "en": "Diarisation failed on the GPU ({error}), new attempt on the CPU..."},
    "fewer_speakers": {"fr": "Seulement {found} locuteur(s) détecté(s) pour {expected} annoncé(s).",
                       "en": "Only {found} speaker(s) detected for {expected} expected."},

    "file_info": {"fr": "Fichier : {name} ({size} MB)",
                  "en": "File: {name} ({size} MB)"},
    "duration_info": {"fr": "Durée : {seconds} s ({minutes} min)",
                      "en": "Duration: {seconds} s ({minutes} min)"},
    "step_language": {"fr": "ETAPE 1/4 : LANGUE",
                      "en": "STEP 1/4: LANGUAGE"},
    "multilingual_mode": {"fr": "Mode multilingue : la langue est détectée à chaque segment.",
                          "en": "Multilingual mode: the language is detected for each segment."},
    "language_forced": {"fr": "Langue imposée : {language}",
                        "en": "Language set: {language}"},
    "no_speech_detected": {"fr": "Aucune parole détectée dans le fichier.",
                           "en": "No speech detected in the file."},
    "likely_languages": {"fr": "Langues les plus probables : {languages}",
                         "en": "Most probable languages: {languages}"},
    "uncertain_prompt": {"fr": "Détection incertaine. Code de langue à utiliser (Entrée = {language}) : ",
                         "en": "Uncertain detection. Language code to use (Enter = {language}): "},
    "uncertain_warning": {"fr": "ATTENTION : détection incertaine, '{language}' retenue. Vérifiez la transcription.",
                          "en": "WARNING: uncertain detection, '{language}' selected. Check the transcription."},
    "language_selected": {"fr": "Langue retenue : {language}",
                          "en": "Selected language: {language}"},
    "step_transcription": {"fr": "ETAPE 2/4 : TRANSCRIPTION",
                           "en": "STEP 2/4: TRANSCRIPTION"},
    "words_transcribed": {"fr": "{words} {transcribed}",
                          "en": "{words} {transcribed}"},
    "no_words": {"fr": "Aucun mot transcrit.",
                 "en": "No words transcribed."},
    "diarization_skipped": {"fr": "Diarisation ignorée (1 seul locuteur)",
                            "en": "Diarisation skipped (single speaker)"},
    "step_diarization": {"fr": "ETAPE 3/4 : DIARISATION",
                         "en": "STEP 3/4: DIARISATION"},
    "speakers_kept": {"fr": "{count} locuteur(s) retenu(s)",
                      "en": "{count} speaker(s) kept"},
    "diarization_failed": {"fr": "La diarisation a échoué ({error}).",
                           "en": "Diarisation failed ({error})."},
    "diarization_failed_console": {"fr": "ATTENTION : {warning} La transcription est enregistrée sans distinction de "
                                         "locuteurs.",
                                   "en": "WARNING: {warning} The transcription is saved without speaker distinction."},
    "step_attribution": {"fr": "ETAPE 4/4 : ATTRIBUTION DES MOTS",
                         "en": "STEP 4/4: WORD ATTRIBUTION"},
    "turn_one": {"fr": "intervention",
                 "en": "turn"},
    "turn_many": {"fr": "interventions",
                  "en": "turns"},
    "more_turn_one": {"fr": "autre intervention",
                      "en": "more turn"},
    "more_turn_many": {"fr": "autres interventions",
                       "en": "more turns"},
    "result_turns": {"fr": "Résultat : {turns}",
                     "en": "Result: {turns}"},
    "speaker_stats": {"fr": "Locuteur {speaker} : {turns}, {minutes} min",
                      "en": "Speaker {speaker}: {turns}, {minutes} min"},
    "preview_title": {"fr": "APERCU (10 premières interventions)",
                      "en": "PREVIEW (first 10 turns)"},
    "speaker_heading": {"fr": "Locuteur {speaker} :",
                        "en": "Speaker {speaker}:"},
    "more_turns": {"fr": "... et {turns}",
                   "en": "... and {turns}"},
    "multilingual": {"fr": "multilingue",
                     "en": "multilingual"},
    "file_header": {"fr": "TRANSCRIPTION",
                    "en": "TRANSCRIPTION"},
    "file_line": {"fr": "Fichier : {name}",
                  "en": "File: {name}"},
    "file_language_speakers": {"fr": "Langue : {language}, Locuteurs : {speakers}",
                               "en": "Language: {language}, Speakers: {speakers}"},
    "speakers_failed": {"fr": "non distingués (échec de la diarisation)",
                        "en": "not distinguished (diarisation failed)"},
    "speaker_label": {"fr": "Locuteur {speaker}",
                      "en": "Speaker {speaker}"},
    "output_paths": {"fr": "TXT : {txt}\nCSV : {csv}",
                     "en": "TXT: {txt}\nCSV: {csv}"},

    "config_title": {"fr": "CONFIGURATION",
                     "en": "CONFIGURATION"},
    "models_available": {"fr": "Modèles disponibles :",
                         "en": "Available models:"},
    "model_large": {"fr": "1. large-v3 - Précision maximale de Whisper, recommandé pour toutes les langues (par défaut)",
                    "en": "1. large-v3 - Highest Whisper accuracy, recommended for all languages (default)"},
    "model_turbo": {"fr": "2. turbo    - Plus rapide, mais moins précis, surtout hors anglais",
                    "en": "2. turbo    - Faster, but less accurate, especially for languages other than English"},
    "model_prompt": {"fr": "Modèle (1 ou 2, Entrée = 1) : ",
                     "en": "Model (1 or 2, Enter = 1): "},
    "model_selected": {"fr": "Modèle sélectionné : {model}",
                       "en": "Selected model: {model}"},
    "import_mode_title": {"fr": "MODE D'IMPORT",
                          "en": "IMPORT MODE"},
    "mode_single": {"fr": "1. Fichier unique",
                    "en": "1. Single file"},
    "mode_batch": {"fr": "2. Import en masse (même configuration pour tous les fichiers)",
                   "en": "2. Batch import (same configuration for all files)"},
    "mode_prompt": {"fr": "Mode (1 ou 2, Entrée = 1) : ",
                    "en": "Mode (1 or 2, Enter = 1): "},
    "drive_title": {"fr": "SAUVEGARDE GOOGLE DRIVE (fortement recommandée en mode bulk)",
                    "en": "GOOGLE DRIVE BACKUP (strongly recommended in batch mode)"},
    "drive_explanation": {"fr": "Chaque transcription est copiée dans MyDrive/transcriptions/ dès qu'elle est terminée.",
                          "en": "Each transcription is copied to MyDrive/transcriptions/ as soon as it is finished."},
    "yes_letter": {"fr": "o",
                   "en": "y"},
    "drive_prompt": {"fr": "Activer la sauvegarde Drive ? (o/n, défaut = {default}) : ",
                     "en": "Enable Drive backup? (y/n, default = {default}): "},
    "speakers_title": {"fr": "NOMBRE DE LOCUTEURS",
                       "en": "NUMBER OF SPEAKERS"},
    "speakers_option_one": {"fr": "- 1 : pas de diarisation",
                            "en": "- 1: no diarisation"},
    "speakers_option_fixed": {"fr": "- 2 à {max} : nombre de locuteurs imposé",
                              "en": "- 2 to {max}: fixed number of speakers"},
    "speakers_option_auto": {"fr": "- vide : détection automatique (recommandé si le nombre est incertain)",
                             "en": "- empty: automatic detection (recommended if the number is uncertain)"},
    "speakers_prompt": {"fr": "Nombre de locuteurs : ",
                        "en": "Number of speakers: "},
    "speakers_max": {"fr": "Le modèle distingue au maximum {max} locuteurs. Les locuteurs au-delà seront confondus avec "
                           "d'autres.",
                     "en": "The model distinguishes at most {max} speakers. Additional speakers will be merged with "
                           "others."},
    "speakers_invalid": {"fr": "Entrez un nombre valide ou laissez vide.",
                         "en": "Enter a valid number or leave empty."},
    "language_title": {"fr": "LANGUE",
                       "en": "LANGUAGE"},
    "language_option_auto": {"fr": "- vide : détection automatique sur les passages de parole (recommandé)",
                             "en": "- empty: automatic detection on speech passages (recommended)"},
    "language_option_code": {"fr": "- code ISO (fr, en, de, es, ...) : langue imposée",
                             "en": "- ISO code (fr, en, de, es, ...): fixed language"},
    "language_option_multi": {"fr": "- multi : plusieurs langues dans le même enregistrement",
                              "en": "- multi: several languages in the same recording"},
    "language_prompt": {"fr": "Langue : ",
                        "en": "Language: "},
    "language_unknown": {"fr": "Code de langue inconnu. Exemples : fr, en, de, es, it, pt, nl, ar, zh.",
                         "en": "Unknown language code. Examples: fr, en, de, es, it, pt, nl, ar, zh."},
    "context_title": {"fr": "CONTEXTE ET VOCABULAIRE (optionnels)",
                      "en": "CONTEXT AND VOCABULARY (optional)"},
    "context_explanation": {"fr": "Le contexte décrit l'enregistrement en une phrase, dans la langue de l'audio.",
                            "en": "The context describes the recording in one sentence, in the language of the audio."},
    "context_example": {"fr": "Exemple : 'Entretien semi-directif avec une élue municipale sur la politique du logement.'",
                        "en": "Example: 'Semi-structured interview with a local councillor about housing policy.'"},
    "context_prompt": {"fr": "Contexte (vide par défaut) : ",
                       "en": "Context (empty by default): "},
    "terms_explanation": {"fr": "Les termes sont les noms propres, sigles et mots techniques attendus, séparés par des "
                                "virgules.",
                          "en": "Terms are the expected proper names, acronyms and technical words, separated by "
                                "commas."},
    "terms_example": {"fr": "Exemple : 'Sciences Po Bordeaux, Konstanz, PLU, Mérignac'",
                      "en": "Example: 'Sciences Po Bordeaux, Konstanz, PLU, Mérignac'"},
    "terms_prompt": {"fr": "Termes (vide par défaut) : ",
                     "en": "Terms (empty by default): "},
    "summary_title": {"fr": "RECAPITULATIF",
                      "en": "SUMMARY"},
    "summary_mode": {"fr": "Mode : {mode}",
                     "en": "Mode: {mode}"},
    "mode_batch_name": {"fr": "Import en masse",
                        "en": "Batch import"},
    "mode_single_name": {"fr": "Fichier unique",
                         "en": "Single file"},
    "summary_model": {"fr": "Modèle : {model}",
                      "en": "Model: {model}"},
    "summary_language": {"fr": "Langue : {language}",
                         "en": "Language: {language}"},
    "summary_speakers": {"fr": "Locuteurs : {speakers}",
                         "en": "Speakers: {speakers}"},
    "auto_detection": {"fr": "détection automatique",
                       "en": "automatic detection"},
    "summary_context": {"fr": "Contexte : {context}, Termes : {terms}",
                        "en": "Context: {context}, Terms: {terms}"},
    "yes": {"fr": "oui",
            "en": "yes"},
    "no": {"fr": "non",
           "en": "no"},
    "summary_drive": {"fr": "Sauvegarde Drive : {drive}",
                      "en": "Drive backup: {drive}"},
    "press_enter": {"fr": "Appuyez sur Entrée pour continuer...",
                    "en": "Press Enter to continue..."},
    "upload_batch": {"fr": "Upload des fichiers (Ctrl/Cmd+clic pour en sélectionner plusieurs) :",
                     "en": "Upload the files (Ctrl/Cmd+click to select several):"},
    "upload_single": {"fr": "Upload du fichier :",
                      "en": "Upload the file:"},
    "no_upload": {"fr": "Aucun fichier uploadé, on recommence.",
                  "en": "No file uploaded, starting again."},
    "loading_model": {"fr": "Chargement du modèle {model}...",
                      "en": "Loading the {model} model..."},
    "file_banner": {"fr": "# FICHIER {index}/{total} : {name}",
                    "en": "# FILE {index}/{total}: {name}"},
    "file_error": {"fr": "ERREUR sur {name} : {error}",
                   "en": "ERROR on {name}: {error}"},
    "batch_summary_title": {"fr": "BILAN DU LOT",
                            "en": "BATCH SUMMARY"},
    "batch_success_count": {"fr": "Fichiers traités avec succès : {done}/{total}",
                            "en": "Files processed successfully: {done}/{total}"},
    "batch_ko": {"fr": "KO - {name} : {error}",
                 "en": "KO - {name}: {error}"},
    "new_transcription_prompt": {"fr": "Nouvelle transcription ? (o/n) : ",
                                 "en": "New transcription? (y/n): "},
    "end": {"fr": "Fin.",
            "en": "End."},
}


def t(key, **values):
    return TEXTS[key][LANG].format(**values)


print("Langue de l'interface / Interface language")
print("1. Français")
print("2. English")
LANG = "en" if input("\n(1 / 2, Entrée/Enter = 1) : ").strip() == "2" else "fr"
print()


def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def _transformers_compatible():
    spec = importlib.util.find_spec("transformers")
    if spec is None or not spec.submodule_search_locations:
        return False
    directory = list(spec.submodule_search_locations)[0]
    return os.path.isdir(os.path.join(directory, "models", "nemotron3_diarization"))


if not (importlib.util.find_spec("faster_whisper") and _transformers_compatible()):
    already_loaded = "transformers" in sys.modules
    print(t("installing"))
    subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    _pip("faster-whisper", "soundfile", "accelerate")
    _pip("git+https://github.com/huggingface/transformers")
    print(t("install_done") + "\n")
    if already_loaded:
        raise SystemExit(t("old_transformers"))
else:
    print(t("deps_present") + "\n")


def preload_cublas12():
    import ctypes
    import glob
    import site

    try:
        ctypes.CDLL("libcublas.so.12", mode=ctypes.RTLD_GLOBAL)
        return True
    except OSError:
        pass

    def find_cublas_dir():
        for directory in site.getsitepackages() + [site.getusersitepackages()]:
            found = glob.glob(os.path.join(directory, "nvidia", "**", "libcublasLt.so.12"), recursive=True)
            if found:
                return os.path.dirname(found[0])
        return None

    directory = find_cublas_dir()
    if directory is None:
        print(t("installing_cublas"))
        _pip("nvidia-cublas-cu12")
        directory = find_cublas_dir()
    if directory is None:
        print(t("cublas_missing"))
        return False
    for name in ("libcublasLt.so.12", "libcublas.so.12"):
        ctypes.CDLL(os.path.join(directory, name), mode=ctypes.RTLD_GLOBAL)
    print(t("cublas_loaded", path=directory) + "\n")
    return True


preload_cublas12()

import base64
import csv
import html
import shutil
import traceback
import uuid
import zipfile

import numpy as np
import soundfile as sf
import torch
from tqdm.auto import tqdm
from IPython.display import display, HTML
from google.colab import files

from faster_whisper import WhisperModel
from faster_whisper.vad import get_speech_timestamps, VadOptions

from transformers import AutoModelForAudioFrameClassification, AutoProcessor
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()

try:
    from faster_whisper.tokenizer import _LANGUAGE_CODES as LANGUAGE_CODES
except ImportError:
    LANGUAGE_CODES = None

print(t("imports_ok") + "\n")

SAMPLE_RATE = 16000
DIARIZATION_MODEL = "nvidia/Nemotron-3-Diarization"
MAX_DIARIZATION_SPEAKERS = 8
ACTIVITY_THRESHOLD = 0.5
MIN_TURN_DURATION = 0.6
SENTENCE_END_PAUSE = 1.5
LANGUAGE_CONFIDENCE_THRESHOLD = 0.7
SENTENCE_ENDINGS = (".", "?", "!", "…", "。", "？", "！")
YES_ANSWERS = ("o", "oui", "y", "yes")
GPU = torch.cuda.is_available()

if not GPU:
    print(t("no_gpu"))
    print(t("no_gpu_slow") + "\n")


ICONS = {
    "download": '<path d="M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4"/>'
                '<polyline points="7 10 12 15 17 10"/><line x1="12" x2="12" y1="15" y2="3"/>',
    "check": '<path d="M20 6 9 17l-5-5"/>',
    "alert": '<circle cx="12" cy="12" r="10"/><line x1="12" x2="12" y1="8" y2="12"/>'
             '<line x1="12" x2="12.01" y1="16" y2="16"/>',
}

PANEL_STYLE = """<style>
.qq-panel {
  --qq-surface: #F4F6F8; --qq-text: #1A2330; --qq-muted: #4A5563;
  --qq-accent: #1F4E79; --qq-on-accent: #FFFFFF; --qq-accent-soft: #DCE6F0;
  --qq-success: #1E6B3A; --qq-warning: #8A4B00; --qq-error: #A3261E;
  color-scheme: light;
  box-sizing: border-box; max-width: 640px; margin: 16px 0; padding: 24px;
  background: var(--qq-surface); color: var(--qq-text);
  font-family: system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
  font-size: 14px; line-height: 20px;
}
@media (prefers-color-scheme: dark) {
  .qq-panel {
    --qq-surface: #1C232B; --qq-text: #E6EBF0; --qq-muted: #A9B4C0;
    --qq-accent: #9CC3E8; --qq-on-accent: #10273D; --qq-accent-soft: #2A3644;
    --qq-success: #8FD1A5; --qq-warning: #F0B35A; --qq-error: #F29B93;
    color-scheme: dark;
  }
}
.qq-panel * { box-sizing: border-box; }
.qq-title { margin: 0; font-size: 16px; line-height: 24px; font-weight: 600; overflow-wrap: anywhere; }
.qq-meta { margin: 4px 0 0; color: var(--qq-muted); }
.qq-actions { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 16px; }
.qq-button {
  display: inline-flex; align-items: center; gap: 8px; min-height: 40px; padding: 8px 16px;
  font-weight: 600; text-decoration: none;
}
.qq-button-primary { background: var(--qq-accent); color: var(--qq-on-accent); }
.qq-button-secondary { background: var(--qq-accent-soft); color: var(--qq-accent); }
.qq-button:hover { text-decoration: underline; text-underline-offset: 4px; }
.qq-button:focus-visible { outline: 2px solid var(--qq-accent); outline-offset: 2px; }
.qq-size { font-weight: 400; }
.qq-note { display: flex; gap: 8px; align-items: flex-start; margin: 16px 0 0; }
.qq-list { list-style: none; display: grid; gap: 8px; margin: 24px 0 0; padding: 0; }
.qq-list li { display: flex; gap: 8px; align-items: flex-start; }
.qq-icon { flex: none; margin-top: 2px; }
.qq-name { font-weight: 600; overflow-wrap: anywhere; }
.qq-hint { display: block; margin-top: 4px; color: var(--qq-muted); }
.qq-ok { color: var(--qq-success); }
.qq-warning { color: var(--qq-warning); }
.qq-error { color: var(--qq-error); }
</style>"""

FILE_LABEL_KEYS = {".txt": "download_text", ".csv": "download_spreadsheet"}
MIME_TYPES = {".txt": "text/plain", ".csv": "text/csv", ".zip": "application/zip"}


def _icon(name, css_class=""):
    return (f'<svg class="qq-icon {css_class}" width="16" height="16" viewBox="0 0 24 24" fill="none" '
            f'stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" '
            f'aria-hidden="true" focusable="false">{ICONS[name]}</svg>')


def _readable_size(num_bytes):
    if num_bytes < 1024 * 1024:
        return t("size_kb", value=max(1, round(num_bytes / 1024)))
    return t("size_mb", value=f"{num_bytes / (1024 * 1024):.1f}".replace(".", t("decimal_separator")))


def _number(n):
    return f"{n:,}".replace(",", t("thousands_separator"))


def _word(n, singular, plural):
    is_plural = n > 1 if LANG == "fr" else n != 1
    return t(plural if is_plural else singular)


def _count(n, singular, plural):
    return f"{_number(n)} {_word(n, singular, plural)}"


def _button(path, primary=False, label=None):
    with open(path, "rb") as f:
        content = f.read()
    ext = os.path.splitext(path)[1]
    label = label or t(FILE_LABEL_KEYS.get(ext, "download_file"))
    css_class = "qq-button-primary" if primary else "qq-button-secondary"
    b64 = base64.b64encode(content).decode()
    return (f'<a class="qq-button {css_class}" download="{html.escape(os.path.basename(path))}" '
            f'href="data:{MIME_TYPES.get(ext, "application/octet-stream")};base64,{b64}">'
            f'{_icon("download")}<span>{label} '
            f'<span class="qq-size">({ext}, {_readable_size(len(content))})</span></span></a>')


def _note(icon, text, icon_class):
    return f'<p class="qq-note">{_icon(icon, icon_class)}<span>{text}</span></p>'


def _drive_note(status):
    if status == "ok":
        return _note("check", t("drive_ok"), "qq-ok")
    start = t("drive_failed") if status == "failed" else t("drive_none")
    return _note("alert", f"{start} {t('drive_download_advice')}", "qq-warning")


def _panel(content, title_id):
    display(HTML(f'{PANEL_STYLE}<section class="qq-panel" lang="{LANG}" '
                 f'aria-labelledby="{title_id}">{content}</section>'))


def _name_span(name):
    return f'<span class="qq-name">{html.escape(name)}</span>'


def error_hint(message):
    m = message.lower()
    if t("no_speech_transcribed") in m:
        return t("hint_no_speech")
    if "ffmpeg" in m:
        return t("hint_ffmpeg")
    if "libcublas" in m or "libcudnn" in m:
        return t("hint_cuda")
    if "out of memory" in m:
        return t("hint_oom")
    return t("hint_default")


def show_file_result(name, outputs, summary, drive_status, position=None):
    title_id = f"qq-{uuid.uuid4().hex[:8]}"
    title = t("result_title", name=html.escape(name))
    if position:
        title += " " + t("position", index=position[0], total=position[1])
    speakers = (t("speakers_undistinguished") if summary.get("warning")
                else _count(summary["speakers"], "speaker_one", "speaker_many"))
    meta = t("result_meta", minutes=max(1, round(summary["duration"] / 60)),
             language=html.escape(str(summary["language"])), speakers=speakers,
             words=_count(summary["words"], "word_one", "word_many"))
    alert = ""
    if summary.get("warning"):
        alert = _note("alert", t("diarization_failed_note"), "qq-warning")
    buttons = "".join(_button(p, primary=(i == 0)) for i, p in enumerate(outputs) if os.path.exists(p))
    _panel(f'<h3 class="qq-title" id="{title_id}">{title}</h3>'
           f'<p class="qq-meta">{meta}</p>'
           f'{alert}<div class="qq-actions">{buttons}</div>'
           f'{_drive_note(drive_status)}', title_id)


def show_file_failure(name, message, position=None):
    title_id = f"qq-{uuid.uuid4().hex[:8]}"
    title = t("failure_title", name=html.escape(name))
    if position:
        title += " " + t("position", index=position[0], total=position[1])
    detail = html.escape(message if len(message) <= 200 else message[:200] + "...")
    _panel(f'<h3 class="qq-title" id="{title_id}">{title}</h3>'
           f'{_note("alert", error_hint(message), "qq-error")}'
           f'<p class="qq-meta">{t("technical_detail", detail=detail)}</p>', title_id)


def show_batch_summary(succeeded, failed, outputs, drive_status):
    title_id = f"qq-{uuid.uuid4().hex[:8]}"
    total = len(succeeded) + len(failed)
    title = t("batch_title", done=len(succeeded), total=total,
              files=_word(total, "file_one", "file_many"),
              transcribed=_word(len(succeeded), "transcribed_one", "transcribed_many"))

    actions = ""
    if outputs:
        archive = t("batch_archive")
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
            for path in outputs:
                if os.path.exists(path):
                    z.write(path, os.path.basename(path))
        actions = f'<div class="qq-actions">{_button(archive, primary=True, label=t("download_all"))}</div>'
        actions += _drive_note(drive_status)

    items = "".join(
        f'<li>{_icon("check", "qq-ok")}<span>{t("item_transcribed", name=_name_span(f))}</span></li>'
        for f in succeeded)
    items += "".join(
        f'<li>{_icon("alert", "qq-error")}<span>{t("item_failed", name=_name_span(f))}'
        f'<span class="qq-hint">{error_hint(err)}</span></span></li>'
        for f, err in failed)

    _panel(f'<h3 class="qq-title" id="{title_id}">{title}</h3>{actions}'
           f'<ul class="qq-list">{items}</ul>', title_id)


def setup_drive(target_folder="transcriptions"):
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive"):
            drive.mount("/content/drive")
        path = f"/content/drive/MyDrive/{target_folder}"
        os.makedirs(path, exist_ok=True)
        print(t("drive_enabled", path=path))
        return path
    except Exception as e:
        print(t("drive_mount_failed", error=e))
        return None


def save_to_drive(paths, drive_folder):
    if not drive_folder:
        return "disabled"
    status = "ok"
    for path in paths:
        if os.path.exists(path):
            try:
                shutil.copy2(path, drive_folder)
                print("  " + t("drive_copied", name=os.path.basename(path)))
            except Exception as e:
                print("  " + t("drive_copy_failed", path=path, error=e))
                status = "failed"
    return status


def format_timestamp(seconds):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


def convert_audio(input_path, output_path="audio_16k.wav"):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", input_path,
                    "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE), "-c:a", "pcm_s16le", output_path], check=True)
    audio, _ = sf.read(output_path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio


def _language_probabilities(model, clip):
    try:
        _, _, probabilities = model.detect_language(clip)
        return probabilities
    except (AttributeError, TypeError):
        _, info = model.transcribe(clip, beam_size=1, without_timestamps=True)
        return info.all_language_probs or [(info.language, info.language_probability)]


def detect_language(model, audio, num_windows=5, window_duration=30):
    timestamps = get_speech_timestamps(audio, VadOptions(min_silence_duration_ms=500))
    if not timestamps:
        return None, 0.0, []
    speech = np.concatenate([audio[ts["start"]:ts["end"]] for ts in timestamps])
    size = window_duration * SAMPLE_RATE
    num_possible = max(1, len(speech) // size)
    indices = np.unique(np.linspace(0, num_possible - 1, min(num_windows, num_possible)).astype(int))

    scores = {}
    for i in indices:
        clip = speech[i * size:(i + 1) * size]
        for language, p in _language_probabilities(model, clip):
            scores[language] = scores.get(language, 0.0) + p / len(indices)

    ranking = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranking[0][0], ranking[0][1], ranking[:3]


def transcribe(model, audio, language, multilingual, context, terms):
    segments, info = model.transcribe(
        audio,
        language=None if multilingual else language,
        multilingual=multilingual,
        word_timestamps=True,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
        beam_size=5,
        condition_on_previous_text=False,
        hallucination_silence_threshold=2.0,
        initial_prompt=context or None,
        hotwords=terms or None,
    )

    words, segment_bounds = [], []
    with tqdm(total=round(info.duration), unit="s", desc=t("progress_transcription")) as bar:
        for seg in segments:
            if seg.words:
                start = len(words)
                for w in seg.words:
                    words.append({"start": w.start, "end": w.end, "text": w.word, "prob": w.probability})
                segment_bounds.append((start, len(words)))
            bar.update(max(0, min(round(seg.end), bar.total) - bar.n))
        bar.update(bar.total - bar.n)
    return words, segment_bounds, info


_diarizer = globals().get("_diarizer")


def load_diarizer():
    global _diarizer
    if not isinstance(_diarizer, tuple):
        _diarizer = None
    if _diarizer is None:
        print(t("loading_diarizer"))
        processor = AutoProcessor.from_pretrained(DIARIZATION_MODEL)
        model = AutoModelForAudioFrameClassification.from_pretrained(
            DIARIZATION_MODEL, device_map="cuda" if GPU else "cpu")
        model.eval()
        _diarizer = (processor, model)
    return _diarizer


def diarize(audio):
    processor, model = load_diarizer()
    inputs = processor(audio, sampling_rate=SAMPLE_RATE)

    def run_model(m):
        with torch.inference_mode():
            return m(**inputs.to(m.device, dtype=m.dtype)).logits

    try:
        logits = run_model(model)
    except RuntimeError as e:
        if not GPU:
            raise
        print(t("diarization_gpu_retry", error=e))
        torch.cuda.empty_cache()
        model.to("cpu")
        logits = run_model(model)

    matrix = logits[0].float().cpu().numpy()
    if matrix.min() < 0 or matrix.max() > 1:
        matrix = 1.0 / (1.0 + np.exp(-matrix))
    return matrix, 0.01


def select_channels(matrix, step, num_speakers):
    activity = (matrix > ACTIVITY_THRESHOLD).sum(axis=0) * step
    channels = [int(c) for c in np.argsort(-activity) if activity[c] >= 1.0]
    if num_speakers:
        if len(channels) < num_speakers:
            print(t("fewer_speakers", found=len(channels), expected=num_speakers))
        channels = channels[:num_speakers]
    return channels or [int(np.argmax(activity))], activity


def assign_speakers(words, matrix, step, channels):
    sub = matrix[:, channels]
    num_frames = sub.shape[0]
    active = np.where(sub.max(axis=1) > ACTIVITY_THRESHOLD)[0]

    for w in words:
        i0 = min(int(w["start"] / step), num_frames - 1)
        i1 = min(max(i0 + 1, int(np.ceil(w["end"] / step))), num_frames)
        block = sub[i0:i1]
        if block.max() > ACTIVITY_THRESHOLD:
            w["speaker"] = channels[int(block.mean(axis=0).argmax())]
        elif len(active):
            middle = int((w["start"] + w["end"]) / 2 / step)
            pos = np.searchsorted(active, middle)
            nearby = [active[j] for j in (pos - 1, pos) if 0 <= j < len(active)]
            frame = min(nearby, key=lambda f: abs(f - middle))
            w["speaker"] = channels[int(sub[frame].argmax())]
        else:
            w["speaker"] = channels[0]
    return words


def split_sentences(words):
    sentences, current = [], []
    for i, w in enumerate(words):
        current.append(w)
        following = words[i + 1] if i + 1 < len(words) else None
        if (w["text"].strip().endswith(SENTENCE_ENDINGS) or following is None
                or following["start"] - w["end"] > SENTENCE_END_PAUSE):
            sentences.append(current)
            current = []
    return sentences


def _runs(sentence):
    runs, start = [], 0
    for i in range(1, len(sentence) + 1):
        if i == len(sentence) or sentence[i]["speaker"] != sentence[start]["speaker"]:
            runs.append((start, i - 1))
            start = i
    return runs


def smooth_turns(sentences):
    for sentence in sentences:
        while True:
            runs = _runs(sentence)
            if len(runs) <= 1:
                break
            durations = [sentence[end]["end"] - sentence[start]["start"] for start, end in runs]
            k = int(np.argmin(durations))
            if durations[k] >= MIN_TURN_DURATION:
                break
            adjacent = [j for j in (k - 1, k + 1) if 0 <= j < len(runs)]
            v = max(adjacent, key=lambda j: durations[j])
            speaker = sentence[runs[v][0]]["speaker"]
            for idx in range(runs[k][0], runs[k][1] + 1):
                sentence[idx]["speaker"] = speaker
    return sentences


def group_turns(words):
    turns = []
    for w in words:
        if turns and turns[-1]["speaker"] == w["speaker"]:
            turns[-1]["end"] = w["end"]
            turns[-1]["text"] += w["text"]
        else:
            turns.append({"speaker": w["speaker"], "start": w["start"], "end": w["end"], "text": w["text"]})
    for turn in turns:
        turn["text"] = turn["text"].strip()
    return turns


def process_file(audio_file, whisper_model, num_speakers, language_mode, context, terms, interactive):
    print(t("file_info", name=audio_file, size=f"{os.path.getsize(audio_file) / (1024 * 1024):.2f}") + "\n")
    audio = convert_audio(audio_file)
    duration = len(audio) / SAMPLE_RATE
    print(t("duration_info", seconds=f"{duration:.1f}", minutes=f"{duration / 60:.1f}") + "\n")

    print("=" * 60)
    print(t("step_language"))
    print("=" * 60)
    multilingual = language_mode == "multi"
    language = None
    if multilingual:
        print(t("multilingual_mode") + "\n")
    elif language_mode:
        language = language_mode
        print(t("language_forced", language=language) + "\n")
    else:
        language, probability, top = detect_language(whisper_model, audio)
        if language is None:
            print(t("no_speech_detected"))
            return [], None
        print(t("likely_languages", languages=", ".join(f"{code} ({p:.0%})" for code, p in top)))
        if probability < LANGUAGE_CONFIDENCE_THRESHOLD:
            if interactive:
                answer = input(t("uncertain_prompt", language=language)).strip().lower()
                language = answer or language
            else:
                print(t("uncertain_warning", language=language))
        print(t("language_selected", language=language) + "\n")

    print("=" * 60)
    print(t("step_transcription"))
    print("=" * 60)
    words, segment_bounds, info = transcribe(whisper_model, audio, language, multilingual, context, terms)
    print(t("words_transcribed", words=_count(len(words), "word_one", "word_many"),
            transcribed=_word(len(words), "transcribed_one", "transcribed_many")) + "\n")
    if not words:
        print(t("no_words"))
        return [], None

    def without_diarization():
        result = []
        for start, end in segment_bounds:
            text = "".join(w["text"] for w in words[start:end]).strip()
            result.append({"speaker": "1", "start": words[start]["start"],
                           "end": words[end - 1]["end"], "text": text})
        return result

    warning = None
    matrix = None
    if num_speakers == 1:
        print(t("diarization_skipped") + "\n")
        turns = without_diarization()
    else:
        print("=" * 60)
        print(t("step_diarization"))
        print("=" * 60)
        try:
            matrix, step = diarize(audio)
            channels, activity = select_channels(matrix, step, num_speakers)
            print(t("speakers_kept", count=len(channels)) + "\n")
        except Exception as e:
            traceback.print_exc()
            warning = t("diarization_failed", error=e)
            print("\n" + t("diarization_failed_console", warning=warning) + "\n")
            turns = without_diarization()
            matrix = None

    if matrix is not None:
        print("=" * 60)
        print(t("step_attribution"))
        print("=" * 60)
        assign_speakers(words, matrix, step, channels)
        sentences = smooth_turns(split_sentences(words))
        words = [w for sentence in sentences for w in sentence]

        order = {}
        for w in words:
            order.setdefault(w["speaker"], len(order) + 1)
        for w in words:
            w["speaker"] = str(order[w["speaker"]])
        turns = group_turns(words)

    speakers = sorted({turn["speaker"] for turn in turns}, key=int)
    print(t("result_turns", turns=_count(len(turns), "turn_one", "turn_many")))
    for speaker in speakers:
        speaker_turns = [turn for turn in turns if turn["speaker"] == speaker]
        total_time = sum(turn["end"] - turn["start"] for turn in speaker_turns)
        print(t("speaker_stats", speaker=speaker, turns=_count(len(speaker_turns), "turn_one", "turn_many"),
                minutes=f"{total_time / 60:.1f}"))

    print("\n" + "=" * 60)
    print(t("preview_title"))
    print("=" * 60)
    for turn in turns[:10]:
        text = turn["text"][:200] + "..." if len(turn["text"]) > 200 else turn["text"]
        print(f"\n[{format_timestamp(turn['start'])} --> {format_timestamp(turn['end'])}] "
              f"{t('speaker_heading', speaker=turn['speaker'])}\n  {text}")
    if len(turns) > 10:
        print("\n" + t("more_turns", turns=_count(len(turns) - 10, "more_turn_one", "more_turn_many")))

    base = os.path.splitext(audio_file)[0]
    displayed_language = t("multilingual") if multilingual else language

    txt_file = f"{base}_transcription.txt"
    with open(txt_file, "w", encoding="utf-8") as f:
        f.write(t("file_header") + "\n")
        f.write(t("file_line", name=audio_file) + "\n")
        f.write(t("duration_info", seconds=f"{duration:.1f}", minutes=f"{duration / 60:.1f}") + "\n")
        f.write(t("file_language_speakers", language=displayed_language,
                  speakers=t("speakers_failed") if warning else len(speakers)) + "\n")
        f.write("=" * 60 + "\n\n")
        for turn in turns:
            f.write(f"[{format_timestamp(turn['start'])} --> {format_timestamp(turn['end'])}] "
                    f"{t('speaker_heading', speaker=turn['speaker'])}\n")
            f.write(f"{turn['text']}\n\n")

    csv_file = f"{base}_transcription.csv"
    with open(csv_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["start_time", "end_time", "duration", "speaker", "text"])
        for turn in turns:
            writer.writerow([f"{turn['start']:.2f}", f"{turn['end']:.2f}", f"{turn['end'] - turn['start']:.2f}",
                             t("speaker_label", speaker=turn["speaker"]), turn["text"]])

    print("\n" + t("output_paths", txt=txt_file, csv=csv_file))
    if os.path.exists("audio_16k.wav"):
        os.remove("audio_16k.wav")
    summary = {"duration": duration, "language": displayed_language, "speakers": len(speakers),
               "words": len(words), "turns": len(turns), "warning": warning}
    return [txt_file, csv_file], summary


_model_cache = globals().get("_model_cache")
_model_cache_name = globals().get("_model_cache_name")

while True:
    print("=" * 60)
    print(t("config_title"))
    print("=" * 60)
    print("\n" + t("models_available"))
    print(t("model_large"))
    print(t("model_turbo"))
    choice = input("\n" + t("model_prompt")).strip() or "1"
    MODEL_SIZE = "turbo" if choice == "2" else "large-v3"
    print(t("model_selected", model=MODEL_SIZE))

    print("\n" + "-" * 60 + "\n" + t("import_mode_title") + "\n" + "-" * 60)
    print(t("mode_single"))
    print(t("mode_batch"))
    BATCH_MODE = (input("\n" + t("mode_prompt")).strip() or "1") == "2"

    print("\n" + "-" * 60 + "\n" + t("drive_title") + "\n" + "-" * 60)
    print(t("drive_explanation"))
    default = t("yes_letter") if BATCH_MODE else "n"
    drive_choice = input("\n" + t("drive_prompt", default=default)).strip().lower() or default
    drive_folder = setup_drive() if drive_choice in YES_ANSWERS else None

    print("\n" + "-" * 60 + "\n" + t("speakers_title") + "\n" + "-" * 60)
    print(t("speakers_option_one"))
    print(t("speakers_option_fixed", max=MAX_DIARIZATION_SPEAKERS))
    print(t("speakers_option_auto"))
    while True:
        answer = input("\n" + t("speakers_prompt")).strip()
        if answer == "":
            NUM_SPEAKERS = None
            break
        if answer.isdigit() and int(answer) >= 1:
            NUM_SPEAKERS = int(answer)
            if NUM_SPEAKERS > MAX_DIARIZATION_SPEAKERS:
                print(t("speakers_max", max=MAX_DIARIZATION_SPEAKERS))
                NUM_SPEAKERS = MAX_DIARIZATION_SPEAKERS
            break
        print(t("speakers_invalid"))

    print("\n" + "-" * 60 + "\n" + t("language_title") + "\n" + "-" * 60)
    print(t("language_option_auto"))
    print(t("language_option_code"))
    print(t("language_option_multi"))
    while True:
        LANGUAGE_MODE = input("\n" + t("language_prompt")).strip().lower()
        if LANGUAGE_MODE in ("", "multi") or LANGUAGE_CODES is None or LANGUAGE_MODE in LANGUAGE_CODES:
            break
        print(t("language_unknown"))

    print("\n" + "-" * 60 + "\n" + t("context_title") + "\n" + "-" * 60)
    print(t("context_explanation"))
    print(t("context_example"))
    CONTEXT = input("\n" + t("context_prompt")).strip()
    print("\n" + t("terms_explanation"))
    print(t("terms_example"))
    TERMS = input("\n" + t("terms_prompt")).strip()

    print("\n" + "=" * 60 + "\n" + t("summary_title") + "\n" + "=" * 60)
    print(t("summary_mode", mode=t("mode_batch_name") if BATCH_MODE else t("mode_single_name")))
    print(t("summary_model", model=MODEL_SIZE))
    print(t("summary_language", language=LANGUAGE_MODE or t("auto_detection")))
    print(t("summary_speakers", speakers=NUM_SPEAKERS or t("auto_detection")))
    print(t("summary_context", context=t("yes") if CONTEXT else t("no"), terms=t("yes") if TERMS else t("no")))
    print(t("summary_drive", drive=drive_folder or t("no")))
    print("=" * 60)
    input("\n" + t("press_enter"))

    print("\n" + (t("upload_batch") if BATCH_MODE else t("upload_single")))
    audio_files = list(files.upload().keys())
    if not BATCH_MODE:
        audio_files = audio_files[:1]
    if not audio_files:
        print(t("no_upload"))
        continue

    if _model_cache is not None and _model_cache_name == MODEL_SIZE:
        whisper_model = _model_cache
    else:
        print("\n" + t("loading_model", model=MODEL_SIZE))
        whisper_model = WhisperModel(MODEL_SIZE, device="cuda" if GPU else "cpu",
                                     compute_type="float16" if GPU else "int8")
        _model_cache, _model_cache_name = whisper_model, MODEL_SIZE

    all_outputs, succeeded, failed, drive_statuses = [], [], [], []
    total = len(audio_files)
    for idx, audio_file in enumerate(audio_files, 1):
        print("\n" + "#" * 60 + "\n" + t("file_banner", index=idx, total=total, name=audio_file) + "\n"
              + "#" * 60 + "\n")
        position = (idx, total) if total > 1 else None
        try:
            outputs, summary = process_file(audio_file, whisper_model, NUM_SPEAKERS, LANGUAGE_MODE,
                                            CONTEXT, TERMS, interactive=not BATCH_MODE)
            if outputs:
                all_outputs.extend(outputs)
                succeeded.append(audio_file)
                drive_status = save_to_drive(outputs, drive_folder)
                drive_statuses.append(drive_status)
                show_file_result(audio_file, outputs, summary, drive_status, position)
            else:
                failed.append((audio_file, t("no_speech_transcribed")))
                show_file_failure(audio_file, t("no_speech_transcribed"), position)
        except Exception as e:
            print("\n" + t("file_error", name=audio_file, error=e))
            traceback.print_exc()
            failed.append((audio_file, str(e)))
            show_file_failure(audio_file, str(e), position)
        finally:
            if os.path.exists(audio_file):
                os.remove(audio_file)

    print("\n" + "=" * 60 + "\n" + t("batch_summary_title") + "\n" + "=" * 60)
    print(t("batch_success_count", done=len(succeeded), total=total))
    for f in succeeded:
        print(f"  OK - {f}")
    for f, err in failed:
        print("  " + t("batch_ko", name=f, error=err))
    if total > 1:
        if not drive_folder:
            batch_status = "disabled"
        elif drive_statuses and all(s == "ok" for s in drive_statuses):
            batch_status = "ok"
        else:
            batch_status = "failed"
        show_batch_summary(succeeded, failed, all_outputs, batch_status)

    if input("\n" + t("new_transcription_prompt")).strip().lower() not in YES_ANSWERS:
        print(t("end"))
        break
    print("\n" * 3)

### Restarting the runtime / Redémarrage de l'environnement

The next cell restarts the Python runtime, which frees the GPU memory and unloads the models. Use it after a persistent error or a blocked run caused by a corrupted file, then run the transcription cell again.

La cellule suivante redémarre l'environnement d'exécution Python, ce qui libère la mémoire du GPU et décharge les modèles. Elle sert en cas d'erreur persistante ou de blocage après un fichier corrompu. Relancez ensuite la cellule de transcription.

In [ ]:
import os
os.kill(os.getpid(), 9)

## Licence

Required Notice: Copyright (c) 2026 Alexandre Mahieu

This notebook is distributed under the PolyForm Noncommercial License 1.0.0 (https://polyformproject.org/licenses/noncommercial/1.0.0). It permits use, modification and redistribution for any noncommercial purpose, in particular by students, researchers and educational institutions. Any copy, even partial, must keep the "Required Notice" line above and the link to the licence. Commercial use requires a separate licence granted by the author. Only the original text of the licence is legally binding.

Ce notebook est distribué sous la licence PolyForm Noncommercial 1.0.0. Cette licence autorise l'utilisation, la modification et la redistribution du notebook à toute fin non commerciale, notamment par les étudiants, les chercheurs et les établissements d'enseignement. Toute copie, même partielle, doit conserver la ligne « Required Notice » ci-dessus ainsi que le lien vers la licence. Un usage commercial nécessite une licence distincte accordée par l'auteur. Seul le texte original de la licence, en anglais, fait foi.

## Third-party licences / Licences des composants tiers

The interface icons (download, check, alert-circle) come from Lucide (https://lucide.dev). These three icons are derived from the Feather project.

Les icônes de l'interface (download, check, alert-circle) proviennent de Lucide (https://lucide.dev). Ces trois icônes dérivent du projet Feather.

```text
ISC License

Copyright (c) 2026 Lucide Icons and Contributors

Permission to use, copy, modify, and/or distribute this software for any
purpose with or without fee is hereby granted, provided that the above
copyright notice and this permission notice appear in all copies.

THE SOFTWARE IS PROVIDED "AS IS" AND THE AUTHOR DISCLAIMS ALL WARRANTIES
WITH REGARD TO THIS SOFTWARE INCLUDING ALL IMPLIED WARRANTIES OF
MERCHANTABILITY AND FITNESS. IN NO EVENT SHALL THE AUTHOR BE LIABLE FOR
ANY SPECIAL, DIRECT, INDIRECT, OR CONSEQUENTIAL DAMAGES OR ANY DAMAGES
WHATSOEVER RESULTING FROM LOSS OF USE, DATA OR PROFITS, WHETHER IN AN
ACTION OF CONTRACT, NEGLIGENCE OR OTHER TORTIOUS ACTION, ARISING OUT OF
OR IN CONNECTION WITH THE USE OR PERFORMANCE OF THIS SOFTWARE.
```

```text
The MIT License (MIT) (for the icons listed above)

Copyright (c) 2013-present Cole Bemis

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```